# 95662 - Introduction to Machine Learning
##### By: Brenno, Camilla, Kok Soon

## 0. Config and Imports

In [1]:
import os
from pathlib import Path
import json
import pandas as pd
import random
import math
import copy
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"CUDA Available: {torch.cuda.is_available()}")

CUDA Available: True


In [3]:
# Fixing seed to remove randomness, for better comparisons
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

### A. Local Setup

In [4]:
# DATA_DIR = Path('./data') # Path to dataset
# MODEL_DIR = Path('./models') # Path to models

# MODEL_DIR.mkdir(exist_ok=True)

### B. Colab Setup

In [5]:
!pip -q install gdown

DATA_DIR = Path("/content/ML_Exam_Project")
DATA_DIR.mkdir(exist_ok=True)

FOLDER_URL = "https://drive.google.com/drive/folders/17Cklb8rQ2DJt_NoWPCbgqkd16V1MOXiw?usp=sharing"

!gdown --folder "$FOLDER_URL" -O "$DATA_DIR"

Retrieving folder contents
Processing file 1mxYncZ-k5CJVcaMHjpEtsp79OSXVrSR2 test_id.jsonl
Processing file 1utlrU9pwh-Ou1LomUKtZXaB6HemPzI9s test_long.jsonl
Processing file 17YHqluGDYL1izzYFYWON5FzDZZaCcist test_ood.jsonl
Processing file 19yevHfLO23hJ9UEs31F0wa0I9YwESMqP train_augmented.jsonl
Processing file 1uZ9iQSPtoYhlb1Kt2jCigoFePQGsgLRb train.jsonl
Processing file 1vt1B-Optfa19JqtnNSOuL5Ao7tq1ySap validation_augmented.jsonl
Processing file 1knh_LSuwKqZeUn5cbFJ9bfnu03iYQKJ0 validation.jsonl
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1mxYncZ-k5CJVcaMHjpEtsp79OSXVrSR2
To: /content/ML_Exam_Project/test_id.jsonl
100% 222k/222k [00:00<00:00, 85.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1utlrU9pwh-Ou1LomUKtZXaB6HemPzI9s
To: /content/ML_Exam_Project/test_long.jsonl
100% 182k/182k [00:00<00:00, 98.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=17YHq

## 1. Data

### 1.1 File Sanity Check

In [6]:
expected_files = [
    "train.jsonl",
    "validation.jsonl",
    "test_id.jsonl",
    "test_ood.jsonl",
    "test_long.jsonl",
    "train_augmented.jsonl",
    "validation_augmented.jsonl"
]

for filename in expected_files:
    path = DATA_DIR / filename
    assert path.exists(), f"Missing file: {path}"

print("All dataset files found:")
for filename in expected_files:
    print(" -", DATA_DIR / filename)

All dataset files found:
 - /content/ML_Exam_Project/train.jsonl
 - /content/ML_Exam_Project/validation.jsonl
 - /content/ML_Exam_Project/test_id.jsonl
 - /content/ML_Exam_Project/test_ood.jsonl
 - /content/ML_Exam_Project/test_long.jsonl
 - /content/ML_Exam_Project/train_augmented.jsonl
 - /content/ML_Exam_Project/validation_augmented.jsonl


### 1.2 Data Loading

In [7]:
# Function to load jsonl into pandas df object
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return pd.DataFrame(records)

In [8]:
train_df = load_jsonl(DATA_DIR / "train.jsonl")
train_augmented_df = load_jsonl(DATA_DIR / "train_augmented.jsonl")
validation_df = load_jsonl(DATA_DIR / "validation.jsonl")
validation_augmented_df = load_jsonl(DATA_DIR / "validation_augmented.jsonl")
test_id_df = load_jsonl(DATA_DIR / "test_id.jsonl")
test_ood_df = load_jsonl(DATA_DIR / "test_ood.jsonl")
test_long_df = load_jsonl(DATA_DIR / "test_long.jsonl")

### 1.3 Understanding Data

#### 1.3.1 Data shapes

In [9]:
print("Train:", train_df.shape)
print("Train augmented:", train_augmented_df.shape)
print("Validation:", validation_df.shape)
print("Validation augmented:", validation_augmented_df.shape)
print("Test ID:", test_id_df.shape)
print("Test OOD:", test_ood_df.shape)
print("Test long:", test_long_df.shape)

Train: (12000, 6)
Train augmented: (12000, 6)
Validation: (2000, 6)
Validation augmented: (2000, 6)
Test ID: (2000, 6)
Test OOD: (2000, 6)
Test long: (1500, 6)


#### 1.3.2 Data Example

In [10]:
train_df.head()

,id,expression,value,length,operator_count,depth
0,train-00000,4+4+7+3+8-1,25,11,5,5
1,train-00001,5-(6-5)+8+8,20,11,4,5
2,train-00002,5-3+1,3,5,2,3
3,train-00003,1-(1+5+8+7+1),-21,13,5,6
4,train-00004,7+1-(6+5+9),-12,11,4,4


#### 1.3.3 Test for White Space

It is important to test for white spaces to know if it is required to strip them away because whitespaces such as (1 + 1 3) is a wrong mathematical expression. Furthermore, our plan to use a binary tree to linearize mathematical expression may run into issues if we use whitespace to separate terms

In [11]:
def check_for_whitespace(*data_df):
    for i, df in enumerate(data_df, start=1):
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"Argument {i} is not a DataFrame")

    combined_df = pd.concat(data_df, ignore_index=True)

    return combined_df["expression"].astype(str).str.contains(r"\s").any()

In [12]:
print("White Space Exist") if check_for_whitespace(train_df,validation_df,test_id_df,test_ood_df,test_long_df) else print("No Whitespace found")

No Whitespace found


#### 1.3.4 Test for Unary Operators

Unary operators like -5 have to be considered when constructing expression trees, if unary operators exist, we have to preprocess our data in a way where the sign and the number are interpreted as one (i.e. "-5" instead of "- 5")

In [13]:
def check_for_unary_operators(*data_df):
    for i, df in enumerate(data_df, start=1):
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"Argument {i} is not a DataFrame")

    combined_df = pd.concat(data_df, ignore_index=True)

    def has_unary(expr):
        for i, ch in enumerate(expr):
            if ch in "+-":
                if i == 0:
                    return True  # starts with + or -
                prev = expr[i - 1]
                if prev in "(-+":
                    return True
        return False

    return combined_df["expression"].apply(has_unary).any()

In [14]:
print("Unary Operator Exist") if check_for_unary_operators(train_df, validation_df, test_id_df, test_ood_df, test_long_df) else print("No Unary Operator found")

No Unary Operator found


#### 1.3.5 Data Preprocessing and Transformation

In this step, we aim to represent our input data in a different form in an attempt to help our models learn better

##### 1.3.5.1 ExpressionTree, Prefix, Postfix

In [15]:
# Node Structure (Intermediary structure required to build trees)
class Node:
    def __init__(self, value, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right

In [16]:
# ExpressionTree Class (class used to represent mathematical expression)
class ExpressionTree:
    def __init__(self, root_node):
        self.root_node = root_node

    def get_root_node(self):
        return self.root_node

    def to_postfix(self):
        def dfs(node):
            if node is None:
                return []
            return dfs(node.left) + dfs(node.right) + [str(node.value)]

        return " ".join(dfs(self.root_node))

    def to_prefix(self):
        def dfs(node):
            if node is None:
                return []
            return [str(node.value)] + dfs(node.left) + dfs(node.right)

        return " ".join(dfs(self.root_node))

    def get_depth(self):
        def dfs(node, current_depth):
            if node is None:
                return current_depth - 1

            left_depth = dfs(node.left, current_depth + 1)
            right_depth = dfs(node.right, current_depth + 1)

            return max(left_depth, right_depth)

        if self.root_node is None:
            return 0

        return dfs(self.root_node, 1)

    def get_operator_count(self):
        operators = "+-"

        def dfs(node):
            if node is None:
                return 0

            count = 1 if str(node.value) in operators else 0

            count += dfs(node.left)
            count += dfs(node.right)

            return count

        return dfs(self.root_node)

In [17]:
def reduce(operator_stack, operand_stack):
    op = operator_stack.pop()

    right = operand_stack.pop()
    left = operand_stack.pop()

    operand_stack.append(Node(op, left, right))


def build_tree_with_expression(expression):
    subtree_stack = []
    operator_stack = []

    for ch in expression:

        if ch.isnumeric():
            subtree_stack.append(Node(ch))

        elif ch == '(':
            operator_stack.append(ch)

        elif ch == ')':
            while operator_stack[-1] != '(':
                reduce(operator_stack, subtree_stack)

            operator_stack.pop()  # remove '('

        elif ch in '+-':

            while (
                operator_stack
                and operator_stack[-1] != '('
            ):
                reduce(operator_stack, subtree_stack)

            operator_stack.append(ch)

    while operator_stack:
        reduce(operator_stack, subtree_stack)

    root = subtree_stack.pop()
    return ExpressionTree(root)

expression = "5-(6-5)+8+8"
build_tree_with_expression(expression).to_postfix()

'5 6 5 - - 8 + 8 +'

##### 1.3.5.2 Target Value Normalization
Normalizing the target variable (output) can help improve model performance and stability during training, especially with regression tasks. By scaling the target values, the model can learn more efficiently. We will normalize `y_train` and `y_val` using the mean and standard deviation calculated from `y_train`.

In [18]:
# Calculate mean and standard deviation from training targets
y_mean = train_df["value"].mean()
y_std = train_df["value"].std()

print(f"Training target mean: {y_mean:.2f}")
print(f"Training target standard deviation: {y_std:.2f}")

Training target mean: 4.86
Training target standard deviation: 10.59


In [19]:
def normalize_target_value(value,mean,std):
    return (value - mean) / std

def denormalize_target_value(normalized_value, mean, std):
    return normalized_value * std + mean

#### 1.3.6 Applying Transformations

In this step, we apply the previously discussed preprocessing steps

In [20]:
def apply_transformation(df):
    df["normalized_value"] = df["value"].apply(lambda y: normalize_target_value(y, y_mean, y_std))
    df["prefix"] = df["expression"].apply(lambda x: build_tree_with_expression(x).to_prefix())
    df["postfix"] = df["expression"].apply(lambda x: build_tree_with_expression(x).to_postfix())

    return df

In [21]:
train_df = apply_transformation(train_df)
train_augmented_df = apply_transformation(train_augmented_df)
validation_df = apply_transformation(validation_df)
validation_augmented_df = apply_transformation(validation_augmented_df)
test_id_df = apply_transformation(test_id_df)
test_ood_df = apply_transformation(test_ood_df)
test_long_df = apply_transformation(test_long_df)

train_df.head()

,id,expression,value,length,operator_count,depth,normalized_value,prefix,postfix
0,train-00000,4+4+7+3+8-1,25,11,5,5,1.901590,- + + + + 4 4 7 3 8 1,4 4 + 7 + 3 + 8 + 1 -
1,train-00001,5-(6-5)+8+8,20,11,4,5,1.429479,+ + - 5 - 6 5 8 8,5 6 5 - - 8 + 8 +
2,train-00002,5-3+1,3,5,2,3,-0.175696,+ - 5 3 1,5 3 - 1 +
3,train-00003,1-(1+5+8+7+1),-21,13,5,6,-2.441825,- 1 + + + + 1 5 8 7 1,1 1 5 + 8 + 7 + 1 + -
4,train-00004,7+1-(6+5+9),-12,11,4,4,-1.592027,- + 7 1 + + 6 5 9,7 1 + 6 5 + 9 + -


#### 1.3.7 Summary of Data

Here, we create a function to give a summary of our data so we can compare the different data splits

In [22]:
# Function to return summary of data
# It is possible to calculate the summary of multiple data split together by passing them in together
def summary(*data_df):
    for i, df in enumerate(data_df, start=1):
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"Argument {i} is not a DataFrame")

    combined_df = pd.concat(data_df, ignore_index=True)

    result = pd.DataFrame({
        "mean": combined_df.mean(numeric_only=True),
        "min": combined_df.min(numeric_only=True),
        "max": combined_df.max(numeric_only=True),
    })

    # Expecting comparison of data split with different size, so we have to normalise for more meaningful comparison
    result.loc["value", "positive_ratio"] = (combined_df["value"] > 0).mean()
    result.loc["value", "negative_ratio"] = (combined_df["value"] < 0).mean()
    result.loc["value", "zero_ratio"] = (combined_df["value"] == 0).mean()

    mode = combined_df.mode(numeric_only=True).iloc[0]
    result["mode"] = mode

    return result

In [23]:
display(summary(train_df))
display(summary(train_augmented_df))
display(summary(validation_df))
display(summary(validation_augmented_df))
display(summary(test_id_df))
display(summary(test_ood_df))
display(summary(test_long_df))

,mean,min,max,positive_ratio,negative_ratio,zero_ratio,mode
value,4.860750e+00,-36.000000,46.000000,0.65725,0.306167,0.036583,2.000000
length,9.607000e+00,5.000000,19.000000,NaN,NaN,NaN,7.000000
operator_count,3.489917e+00,2.000000,5.000000,NaN,NaN,NaN,4.000000
depth,3.689917e+00,3.000000,6.000000,NaN,NaN,NaN,4.000000
normalized_value,2.072416e-18,-3.858156,3.884453,NaN,NaN,NaN,-0.270118


#### 1.3.8 Helper Function for Plotting Loss Graph

In [25]:
def to_list(x):
    return [i.detach().cpu().item() if torch.is_tensor(i) else i for i in x]

def draw_loss_graph(epochs, train_losses, val_losses = None, val_losses_RMSE=None, val_losses_MAE=None):     #for the real training if we want to plot we don't have a validation set
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, train_losses, label="Train Loss", marker='o')
    if val_losses is not None:
        plt.plot(epochs, to_list(val_losses), label="Validation Loss", marker='o')       #useful for seeing eventual overfitting
        plt.plot(epochs, to_list(val_losses_RMSE), label="Root Mean Squared Error on the validation set", marker='o')
        plt.plot(epochs, to_list(val_losses_MAE), label="Mean Absolute Error on the validation set", marker='o')
    plt.xlabel("Epochs")
    plt.ylabel("Losses")
    plt.legend()
    plt.title("Loss Graph")

## 2. Data Augmentation

In [26]:
def insert_expression(expr1,expr2):
    ALLOWED_REPLACEMENT_CHAR = "0123456789"
    # Identify possible insertion point
    insertion_point = []
    expr1 = list(expr1)
    for i in range(0,len(expr1)):
        if expr1[i] in ALLOWED_REPLACEMENT_CHAR:
            insertion_point.append(i)

    # Insert
    replacement_index = random.choice(insertion_point)
    result = (
        expr1[:replacement_index]
        + list(expr2)
        + expr1[replacement_index+1:]
    )
    return "".join(result)

In [27]:
def generate_additional_expressions(n,target_min, target_max, df):
    expression_list = df.sample(n)["expression"].tolist()
    result = []

    for expression in expression_list:
        while len(expression) < target_max:
            if len(expression) >= target_min:
                if random.random() < 0.5:
                    break

            remaining_space = target_max - len(result) + 1

            candidates = df[df["length"] <= remaining_space]

            if candidates.empty:
                break

            additional_exp = candidates.sample(n=1)["expression"].iloc[0]
            expression = insert_expression(expression, additional_exp)
        result.append(expression)

    return result

In [28]:
def expressions_to_jsonl(expression_list,data_split_name,save_path,start_id=0):
    rows = []
    id = start_id

    for expression in expression_list:
        expression_tree = build_tree_with_expression(expression)

        row = {"id": f"{data_split_name}-{id:05d}",
               "expression": expression,
               "value": eval(expression),
               "length": len(expression),
               "operator_count": expression_tree.get_operator_count(),
               "depth": expression_tree.get_depth()}
        rows.append(row)
        id+=1

    with open(save_path, "w") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")


In [29]:
expression_list = generate_additional_expressions(n=len(train_df),target_min=12, target_max=21, df=train_df)
expressions_to_jsonl(expression_list,"train_augmented",os.path.join(DATA_DIR,"train_augmented.jsonl"))
expression_list = generate_additional_expressions(n=len(validation_df),target_min=12, target_max=21, df=train_df)
expressions_to_jsonl(expression_list,"validation_augmented",os.path.join(DATA_DIR,"validation_augmented.jsonl"))